In [1]:
from google.colab import drive
import os
import pandas as pd
from scipy import stats
import numpy as np

In [2]:
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/data'

# 3. Automatically change the working directory so relative paths work
if os.path.exists(DATA_DIR):
    os.chdir(DATA_DIR)
    print(f"Successfully connected! Current working directory: {os.getcwd()}")
else:
    print(f"Error: The folder '{DATA_DIR}' could not be found.")
    print("Make sure to add a shortcut of the 'data' folder to your 'My Drive'.")

Mounted at /content/drive
Successfully connected! Current working directory: /content/drive/.shortcut-targets-by-id/1_ib6SYLLQ6Asjybns0GSlgBDwFQrnmkj/data


### Uploading Data and Addressing Missing Values and Outliers

## Data Cleaning Methodology

The datasets are cleaned by addressing missing values and potential outliers.

### Missing values

Missing values are handled according to the meaning of the variable rather than
using one imputation method for every column.

The following rules are used:

- Sentinel values representing missing data are converted to `NaN`.
- Rows missing an essential measurement are removed when the observation cannot
  be used for the analysis.
- Short gaps in sequential environmental measurements may be filled using
  linear interpolation when nearby observations provide a reasonable estimate.
- Missing categorical metadata is represented as `"Unknown"` when appropriate.
- Measurement or metadata values that cannot be reliably inferred are left as
  `NaN`.
- Median imputation is used only when a numerical variable can reasonably be
  represented by a typical value and interpolation is not appropriate.

### Outliers

The Interquartile Range (IQR) method is used consistently for outlier screening.

For each measurement:

- IQR = Q3 - Q1
- Lower bound = Q1 - 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR

Values outside these bounds are considered statistical outliers. However,
statistical outliers are not automatically removed. They are investigated using
their units, sampling context, location, quality information, and physical
plausibility.

Values are removed only when there is evidence that they are clearly erroneous
or physically implausible.

In [3]:
# Load all datasets
chlorophylla1 = pd.read_csv("ChlorophyllaConcentrationSarasota.txt")
chlorophylla2 = pd.read_csv("ChlorophyllaConcentrationSarasota_Dataset2.txt")
sarasotaWind = pd.read_csv("SarasotaWindData.txt")

/tmp/ipykernel_2590/2205226558.py:2: DtypeWarning: Columns (1,2,3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  chlorophylla1 = pd.read_csv("ChlorophyllaConcentrationSarasota.txt")
/tmp/ipykernel_2590/2205226558.py:3: DtypeWarning: Columns (1,2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  chlorophylla2 = pd.read_csv("ChlorophyllaConcentrationSarasota_Dataset2.txt")
/tmp/ipykernel_2590/2205226558.py:4: DtypeWarning: Columns (1,2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  sarasotaWind = pd.read_csv("SarasotaWindData.txt")


## ChlorophyllaConcentrationSarasota.txt

In [4]:
print("Shape:", chlorophylla1.shape)

Shape: (4202009, 5)


In [5]:
chlorophylla1.head()

,time,altitude,latitude,longitude,chlor_a
0,UTC,m,degrees_north,degrees_east,mg m^-3
1,2020-06-15T12:00:00Z,0.0,27.99375,276.50625,NaN
2,2020-06-15T12:00:00Z,0.0,27.99375,276.54375,NaN
3,2020-06-15T12:00:00Z,0.0,27.99375,276.58125,NaN
4,2020-06-15T12:00:00Z,0.0,27.99375,276.61875,NaN


In [6]:
# Check missing values before cleaning.
print("Missing values:")
print(chlorophylla1.isna().sum())

Missing values:
time               0
altitude           0
latitude           0
longitude          0
chlor_a      3162496
dtype: int64


In [7]:
print("\nColumn names:")
print(chlorophylla1.columns)


Column names:
Index(['time', 'altitude', 'latitude', 'longitude', 'chlor_a'], dtype='object')


In [8]:
chlorophylla1.describe()

,time,altitude,latitude,longitude,chlor_a
count,4202009,4202009.0,4.202009e+06,4.202009e+06,1.039513e+06
unique,2229,3.0,8.300000e+01,9.300000e+01,1.029321e+06
top,2026-09-05T12:00:00Z,0.0,2.656875e+01,2.777437e+02,3.897014e-01
freq,1886,4070937.0,9.931400e+04,8.849900e+04,8.000000e+00


### Chlorophyll-a (1) Missing Values

The chlor_a column contains missing values (NaN) representing locations and times where chlorophyll-a measurements were not available. These missing values were not median-imputed, because replacing them with a median value would create artificial chlorophyll-a measurements for observations where no actual measurement was recorded.

In [9]:
# Convert chlorophyll-a values to numeric.
# Invalid/non-numeric values are converted to NaN.
chlorophylla1["chlor_a"] = pd.to_numeric(chlorophylla1["chlor_a"], errors="coerce")

In [10]:
# Remove columns that contain missing chlorophyll-a measurements.
chlorophylla1 = chlorophylla1.dropna(subset=['chlor_a'])

In [11]:
# Check summary statistics after replacing sentinel values.
chlorophylla1.describe()

,chlor_a
count,1.039512e+06
mean,2.277557e+00
std,4.971443e+00
min,1.000000e-03
25%,2.562752e-01
50%,5.677464e-01
75%,2.074664e+00
max,9.999950e+01


### Chlorophyll-a (1) Outlier Detection

IQR is applied to the chlorophyll-a measurements. The date, altitude, latitude, and longitude variables are not treated as environmental measurement outliers.

IQR outliers are investigated rather than automatically deleted because unusually high chlorophyll-a concentrations may represent legitimate observations.

In [12]:
# Calculate Q1, Q3, and IQR for chlorophyll-a.
Q1 = chlorophylla1["chlor_a"].quantile(0.25)
Q3 = chlorophylla1["chlor_a"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower bound:")
print(lower_bound)

print("\nUpper bound:")
print(upper_bound)

# Identify IQR outliers.
outliers = (
    (chlorophylla1["chlor_a"] < lower_bound) |
    (chlorophylla1["chlor_a"] > upper_bound)
)

print("\nNumber of IQR outliers:")
print(outliers.sum())

Lower bound:
-2.4713077250000004

Upper bound:
4.802246875

Number of IQR outliers:
133506


In [13]:
# Inspect chlorophyll-a outliers.
display(
    chlorophylla1[outliers][
        ["time", "altitude", "latitude", "longitude", "chlor_a"]
    ].sort_values(
        "chlor_a",
        ascending=False
    ).head(20)
)

,time,altitude,latitude,longitude,chlor_a
3640071,2025-11-03T12:00:00Z,0.0,27.95625,278.15625,99.999504
4013309,2026-05-22T12:00:00Z,0.0,26.56875,277.93125,99.779390
3793337,2026-01-23T12:00:00Z,0.0,27.54375,277.93125,99.620750
288627,2020-11-15T12:00:00Z,0.0,27.95625,277.33125,99.581080
361826,2020-12-23T12:00:00Z,0.0,26.71875,277.81875,99.567710
3864364,2026-03-01T12:00:00Z,0.0,26.53125,278.04375,99.277720
3793012,2026-01-23T12:00:00Z,0.0,27.80625,277.81875,99.247010
3470726,2025-08-05T12:00:00Z,0.0,27.61875,277.44375,99.152120
333867,2020-12-09T12:00:00Z,0.0,27.99375,278.15625,99.129616
3799866,2026-01-26T12:00:00Z,0.0,26.83125,277.81875,99.012330


In [14]:
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

Q1: 0.25627524999999995
Q3: 2.0746639
IQR: 1.8183886500000002
Lower bound: -2.4713077250000004
Upper bound: 4.802246875


In [15]:
# Count chlorophyll-a values above the upper IQR bound
high_outliers = chlorophylla1["chlor_a"] > upper_bound

print("Number of high outliers:")
print(high_outliers.sum())

print("Percentage of high outliers:")
print(high_outliers.mean() * 100)

Number of high outliers:
133506
Percentage of high outliers:
12.843141781913051


In [16]:
# Examine the distribution of high chlorophyll-a values
print(
    chlorophylla1["chlor_a"].quantile(
        [0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 1.00]
    )
)

0.750     2.074664
0.900     6.165387
0.950    10.044828
0.990    22.339325
0.995    31.075348
0.999    59.129302
1.000    99.999504
Name: chlor_a, dtype: float64


In [19]:
# Analyze altitude values
print(chlorophylla1["altitude"].unique())

['0.0' 0.0]


In [20]:
# Remove altitude column
chlorophylla1 = chlorophylla1.drop(columns=["altitude"])

### Chlorophyll-a (1) Outlier Assessment

The IQR method identified statistical outliers in the chlorophyll-a measurements. These values were investigated to determine whether they represented data errors or potentially valid chlorophyll-a observations.

The identified outliers were retained because the high chlorophyll-a values form a substantial portion of the upper tail of the distribution rather than appearing to be isolated anomalous observations. The observed extreme chlorophyll-a measurements may therefore represent legitimate observations rather than erroneous data.

In [21]:
os.chdir(DATA_DIR+"/cleaned")
chlorophylla1.to_csv("ChlorophyllaConcentrationSarasota_cleaned.csv", index=False)
print("Saved cleaned dataset to ChlorophyllaConcentrationSarasota_cleaned.csv")


Saved cleaned dataset to ChlorophyllaConcentrationSarasota_cleaned.csv
